In [1]:
import torch

In [2]:
text = open('../data.txt', 'r').read()

In [3]:
text[:100]

'Intelligence (AI) and Machine Learning (ML) are transforming modern healthcare systems by enabling i'

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print('Chars length', vocab_size)

Chars length 64


In [5]:
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for i, s in enumerate(chars)}

encode = lambda sen: [stoi[s] for s in sen]
decode = lambda arr: ''.join([itos[s] for s in arr])


In [6]:
encode('khalid khan'), decode([46, 43, 36, 47, 44, 39, 1, 46, 43, 36, 49])

([46, 43, 36, 47, 44, 39, 1, 46, 43, 36, 49], 'khalid khan')

In [7]:
encode_text = torch.tensor(encode(text))
print(encode_text[:100])

tensor([20, 49, 55, 40, 47, 47, 44, 42, 40, 49, 38, 40,  1,  2, 12, 20,  3,  1,
        36, 49, 39,  1, 24, 36, 38, 43, 44, 49, 40,  1, 23, 40, 36, 53, 49, 44,
        49, 42,  1,  2, 24, 23,  3,  1, 36, 53, 40,  1, 55, 53, 36, 49, 54, 41,
        50, 53, 48, 44, 49, 42,  1, 48, 50, 39, 40, 53, 49,  1, 43, 40, 36, 47,
        55, 43, 38, 36, 53, 40,  1, 54, 60, 54, 55, 40, 48, 54,  1, 37, 60,  1,
        40, 49, 36, 37, 47, 44, 49, 42,  1, 44])


In [8]:
n = int(0.9 * len(encode_text))
train = encode_text[:n]
val = encode_text[n:]

In [9]:
block_size = 8
train[:block_size+1]

tensor([20, 49, 55, 40, 47, 47, 44, 42, 40])

In [10]:
x = train[:block_size]
y = train[1: block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]

    print(f'if the input is context {context} then target is {target}')



if the input is context tensor([20]) then target is 49
if the input is context tensor([20, 49]) then target is 55
if the input is context tensor([20, 49, 55]) then target is 40
if the input is context tensor([20, 49, 55, 40]) then target is 47
if the input is context tensor([20, 49, 55, 40, 47]) then target is 47
if the input is context tensor([20, 49, 55, 40, 47, 47]) then target is 44
if the input is context tensor([20, 49, 55, 40, 47, 47, 44]) then target is 42
if the input is context tensor([20, 49, 55, 40, 47, 47, 44, 42]) then target is 40


In [11]:
torch.manual_seed(123)

batch_size = 4
block_size = 8

def get_batch(split):
    data = train if split == 'train' else val
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i: i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])
    return x, y 

xb, yb = get_batch('train')
print("inputs:")
print(xb.shape)
print(xb)

print("targets: ")
print(yb.shape)
print(yb)


print("---------")


for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f'when input is {context.tolist()} the target: {target}')



inputs:
torch.Size([4, 8])
tensor([[39, 44, 42, 44, 55, 36, 47,  1],
        [44, 55, 43, 48, 54,  1, 36, 49],
        [43, 40, 44, 53,  0, 53, 40, 54],
        [ 1, 36, 39, 39, 44, 55, 44, 50]])
targets: 
torch.Size([4, 8])
tensor([[44, 42, 44, 55, 36, 47,  1, 43],
        [55, 43, 48, 54,  1, 36, 49, 39],
        [40, 44, 53,  0, 53, 40, 54, 51],
        [36, 39, 39, 44, 55, 44, 50, 49]])
---------
when input is [39] the target: 44
when input is [39, 44] the target: 42
when input is [39, 44, 42] the target: 44
when input is [39, 44, 42, 44] the target: 55
when input is [39, 44, 42, 44, 55] the target: 36
when input is [39, 44, 42, 44, 55, 36] the target: 47
when input is [39, 44, 42, 44, 55, 36, 47] the target: 1
when input is [39, 44, 42, 44, 55, 36, 47, 1] the target: 43
when input is [44] the target: 55
when input is [44, 55] the target: 43
when input is [44, 55, 43] the target: 48
when input is [44, 55, 43, 48] the target: 54
when input is [44, 55, 43, 48, 54] the target: 1
when 

In [12]:
# ix = torch.randint(len(encode_text) - block_size, (batch_size, ))
# x = torch.stack([encode_text[i: i+block_size] for i in ix])
# x 

In [ ]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(133)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx) # Batch, Time, Channel 

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            B, T = targets.shape
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):
            
            logits, loss = self.forward(idx)
            
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx
    
# m = BigramLanguageModel(vocab_size)
# logits, loss = m(xb, yb)
# print(logits.shape) #
# print(loss) 

# idxs = m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()
# print(decode(idxs))


torch.Size([32, 64])
tensor(4.3932, grad_fn=<NllLossBackward0>)

D“oz)sRzF9sfdNdyFhGvX6-i.)sMDowrG9sLWX1obdMt
pUINJxB9eXCSAbdu“j(o,“nlCt
fr2h-60r,taM26baTFqmhjyGw6k



In [40]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(113)

n_embd = 32

# Adding the Attention 
class BigramLanguageModelV2(nn.Module):
    def __init__(self):

        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)



    def forward(self, idx, targets=None):

        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)
        

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            B, T = targets.shape
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):
            
            logits, loss = self.forward(idx)
            
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx
    
m = BigramLanguageModelV2()
logits, loss = m(xb, yb)
print(logits.shape) #
print(loss) 

# idxs = m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()
# print(decode(idxs))


torch.Size([256, 64])
tensor(4.5601, grad_fn=<NllLossBackward0>)


In [26]:
lm_head = nn.Linear(12, 1)
n_emb = nn.Embedding(34, 34)
n_emb

Embedding(34, 34)

In [14]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [29]:
batch_size = 32

for steps in range(10000):

    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

loss.item()


4.72006368637085

In [16]:
idxs = m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=300)[0].tolist()
print(decode(idxs))


Kal aksabls pt Pus, s. matysysullicace ap
to isistere ancalenicongermaik”02GBWIC6 In Indd us Rice th dulerereveng 16 masurendutheanel malaplefen
AI.
Teron
FleaizMLanthe s
ginteanss o tuth herond m ofode pl
fenig hendinthindit wel Th pkstes l Machcans prte emes in ealt od AInpornde ffellepused ck ffo


In [17]:
k = torch.randn((4, 8, 64))
k = k.view(k.shape[0] * k.shape[1], k.shape[-1])

t = torch.randn((4, 8))
t = t.view(t.shape[0]*t.shape[1],)
t.shape, k.shape

(torch.Size([32]), torch.Size([32, 64]))

## Mathematical Trick in Self Attention

In [18]:
torch.manual_seed(222)
B, T, C= 4, 8, 2 # batch, time , channel

x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [19]:
# Version 1 
# We want x[b,t] = mean_{i<=t} x[b, t]
xbow = torch.zeros((B,T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, dim=0)

In [20]:
x[0], xbow[0]

(tensor([[-1.4609,  0.7528],
         [ 0.7871,  1.2646],
         [-0.6371, -1.1649],
         [-0.8392, -1.1444],
         [ 0.2068, -0.6295],
         [ 0.7378,  0.2925],
         [-2.2334,  0.0755],
         [ 0.4478, -1.2305]]),
 tensor([[-1.4609,  0.7528],
         [-0.3369,  1.0087],
         [-0.4370,  0.2842],
         [-0.5375, -0.0730],
         [-0.3887, -0.1843],
         [-0.2009, -0.1048],
         [-0.4913, -0.0791],
         [-0.3739, -0.2230]]))

In [21]:
# Version 2 
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ---> (B, T, C)
torch.allclose(xbow, xbow2)

True

In [22]:
# Verisoin 3: Softmax
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
torch.allclose(xbow, x)

False

In [23]:
torch.manual_seed(122)
a = torch.tril(torch.ones(3, 3))
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print('a=')
print(a)
print('---')

print('b=')
print(b)
print('---')

print('c=')
print(c)
print('---')

a=
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
---
b=
tensor([[9., 6.],
        [2., 0.],
        [8., 4.]])
---
c=
tensor([[ 9.,  6.],
        [11.,  6.],
        [19., 10.]])
---


In [44]:
# Verisoin 4: SeflAttention

torch.manual_seed(3443)

B, T, C= 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)

wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) --> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zero((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)

out = wei @ v

out.shape

torch.Size([4, 8, 16])